# Clustering with Outlier Removal Implementation

In [ ]:
import pandas as pd
import numpy as np
import time

from utils.data_utils import *
from utils.cor_utils import *
from utils.evaluation import *
%load_ext autoreload
%autoreload 2


COR Algorithm

In [ ]:
def COR(X: pd.DataFrame, K: int, o: int, r: int, n_init: int = 1, k_max: int = None, additional_bp: list = None, verbose: bool = False, seed: int = None) -> tuple[np.ndarray, float, dict]:
    """
    Main function; runs the COR algorithm as presented in the paper. In this version, if the dataset contains categorical columns,
    it is left possible to use them as additional basic partitions.
    The time spent in each step of the algorithm is also measured.

    Inputs:
    -   X: the data matrix
    -   K: the number of clusters
    -   o: the number of outliers
    -   r: the number of basic partitions generated by K-means
    -   n_init: the number of times we run K-means--
    -   k_max: the maximum number of cluster each basic partition can contain; if not provided, it is set to 2*K
    -   additional_bp: the list of extra basic partitions
    -   verbose: a boolean that, if set to True, enables a log of every iteration; False by default
    -   seed: an integer value for reproducibility; not set by default

    Outputs:
    -   The final partition in the form of a vector of labels
    -   The minimum objective value obtained
    -   Some run statistics (obj_values, corresponding labels, number of iterations and execution time (in seconds) of each function)
    """

    #setup
    if k_max is None:
        k_max = 2*K
    times = {"basic partitions": [], 
             "binary matrices": [], 
             "generate centroids": [],
             "KL distances": [], 
             "extract outliers": [], 
             "update centroids": [], 
             "holoentropy": []}

    start = time.perf_counter()
    basic_partitions_labels = generate_basic_partitions(X, r, k_max=k_max, seed=seed)
    if additional_bp:
        basic_partitions_labels += additional_bp #categorical columns are turned into basic partitions
    times["basic partitions"].append(time.perf_counter() - start)

    start = time.perf_counter()
    B, tilde_B = build_binary_matrices(basic_partitions_labels)
    BB = np.hstack([B, tilde_B])
    times["binary matrices"].append(time.perf_counter() - start)

    best_obj_value = np.inf
    best_labels = None
    run_statistics = {"obj_values": [], "labels": [], "iterations": []}

    for i in range(n_init): #we run K-means-- n_init times
        start = time.perf_counter()
        centroids = generate_centroids(BB, K)
        times["generate centroids"].append(time.perf_counter() - start)

        n_iter = 0
        prev_obj_value = np.inf

        while n_iter < 100:
            n_iter += 1

            start = time.perf_counter()
            c_labels, min_distances = compute_kl_distances(BB, centroids=centroids)
            times["KL distances"].append(time.perf_counter() - start)

            start = time.perf_counter()
            outlier_candidates = extract_outliers(min_distances, o)
            c_labels[outlier_candidates] = -1
            times["extract outliers"].append(time.perf_counter() - start)

            start = time.perf_counter()
            centroids = update_centroids(BB, c_labels, centroids)
            times["update centroids"].append(time.perf_counter() - start)

            start = time.perf_counter()
            obj_value = compute_holoentropy(c_labels, centroids)
            times["holoentropy"].append(time.perf_counter() - start)
            
            if prev_obj_value - obj_value <= 1e-10: #the while cycle terminates once the marginal difference gets too small
                break

            if verbose:
                print(f"Run: {i}. Iteration: {n_iter}; objective value: {obj_value}\n")
            prev_obj_value = obj_value

        if obj_value < best_obj_value:
            best_obj_value = obj_value
            best_labels = c_labels

        run_statistics["iterations"].append(n_iter)
        run_statistics["labels"].append(c_labels)
        run_statistics["obj_values"].append(obj_value)

    times_df = pd.DataFrame({step: pd.Series(step_time).agg(["count", "sum", "mean", "std"]) for step, step_time in times.items()}).T
    times_df.columns = ["calls", "total", "mean", "stddev"]
    run_statistics["times"] = times_df

    return best_labels, best_obj_value, run_statistics

## Datasets and tests
Each test cell runs one configuration; the parameters to change to obtain the other results are commented above the rows where they are used

In [ ]:
n_init = 10 #number of tests per dataset
r = 50
seed = 42

### Statlog Shuttle

In [ ]:
header = ["A1", "A2", "A3", "A4", "A5", "A6", "A7", "A8", "A9", "Class"]
shuttle_train = pd.read_csv("datasets/statlog+shuttle/train_shuttle.csv", sep = ",",header=None, names=header)
shuttle_test  = pd.read_csv("datasets/statlog+shuttle/test_shuttle.csv", sep = ",",header=None, names = header)
shuttle = pd.concat([shuttle_train,shuttle_test], ignore_index=True)
# shuttle.info()
# shuttle.describe()
shuttle_target  = shuttle.Class
shuttle.drop("Class",axis = 1, inplace = True)
# plot_correlation(shuttle)

# r_cols = find_most_redundant_columns(shuttle,1)
# shuttle = shuttle.drop(columns=r_cols)

# -------------------------------------------------

# shuttle_target.value_counts()
shuttle = standardise(shuttle)
outlier_indexes = shuttle[shuttle_target.isin([2, 3, 6, 7])].index   # 2,3,6,7 are the classes considered outlier set by the paper; found with value_counts()
true_outliers_shuttle = np.zeros(len(shuttle), dtype=bool)
true_outliers_shuttle[outlier_indexes] = True


cor_labels_shuttle,shuttle_obj, shuttle_stats = COR(shuttle, K = 3, o=len(true_outliers_shuttle[true_outliers_shuttle==True]),r = r,n_init= n_init, seed = seed)
# PCA_scatter_plot(shuttle,cor_labels_shuttle,true_outliers_shuttle)
shuttle_stats["times"]

### Wine

In [ ]:
red_wine_quality = pd.read_csv("datasets/wine+quality/winequality-red.csv",delimiter=";")
# red_wine_quality.info()
# red_wine_quality.describe()
red_target = red_wine_quality["quality"].to_numpy()
red_wine_quality.drop("quality",axis=1, inplace=True)
# plot_correlation(red_wine_quality)

# -----------------------------------------------------

red_wine_quality = standardise(red_wine_quality)                                                           # o = 0%, 5%, 10%
cor_labels_red_wine, red_wine_obj, red_wine_stats = COR(red_wine_quality, K = len(np.unique(red_target)),o = int(len(red_wine_quality)*0.1), 
                                                        r= r, n_init=n_init, seed = seed)
PCA_scatter_plot(red_wine_quality, cor_labels_red_wine)


In [ ]:
white_wine_quality = pd.read_csv("datasets/wine+quality/winequality-white.csv",delimiter=";")
# white_wine_quality.info()
# white_wine_quality.describe()
white_target = white_wine_quality["quality"].to_numpy()
white_wine_quality.drop("quality",axis = 1, inplace = True)
# plot_correlation(white_wine_quality)

# --------------------------------------------------------

white_wine_quality = standardise(white_wine_quality)                                                                 # o = 0%, 5%, 10%
cor_labels_white_wine, white_wine_obj, white_wine_stats = COR(white_wine_quality, K = len(np.unique(white_target)), o = int(len(white_wine_quality)*0.1), 
                                                              r = r, n_init = n_init, seed = seed)
PCA_scatter_plot(white_wine_quality,labels=cor_labels_white_wine)

### Credit Card Fraud 


In [ ]:
ccf = pd.read_csv("datasets/creditcard.zip", sep=",")
# ccf.info()
# ccf.describe()
# plot_correlation(ccf.drop("Class", axis = 1), show_values=False)

# -------------------------------------------------

frauds = ccf[ccf["Class"] == 1].sample(frac = 1, random_state=seed) #change the frac input to consider less outliers
no_frauds = ccf[ccf["Class"] == 0].sample(25000, random_state=seed)
ccf = pd.concat([no_frauds, frauds]).sample(frac=1, random_state=seed).reset_index(drop=True)

ccf_target = ccf["Class"]
true_outliers_ccf = (ccf_target == 1)
ccf = ccf.drop(columns="Class")

ccf["Amount"] = np.log(1+ccf["Amount"]) #there is too much variability

# -------------------------------------------------

ccf = standardise(ccf)                     #K = 2,3,4
cor_labels_ccf, ccf_obj, ccf_stats = COR(ccf, K= 4, o=len(frauds), r=r, n_init=n_init, seed=seed)

# PCA_scatter_plot(ccf, cor_labels_ccf, true_outliers_ccf)               

### Forestfires


In [ ]:
fires = pd.read_csv("datasets/forestfires.csv")
# fires.info()
# fires.describe()
# plot_correlation(fires)

#---------------------------------------------------

fires, add_parts_fires = make_categoric_partitions(fires)
fires = standardise(fires)                      #K = 2,3,4 ; o = 0%, 5%, 10%
cor_labels_fires,fires_obj, fires_stats = COR(fires, K = 3, o = int(len(fires)*0.05), r = r,n_init=n_init, additional_bp=add_parts_fires, seed = seed)

PCA_scatter_plot(fires, cor_labels_fires)


## Performances Evaluation

In [ ]:
stats_shuttle = runs_statistics(shuttle_stats,shuttle_target,true_outliers_shuttle)
stats_red = runs_statistics(red_wine_stats, red_target)
stats_white = runs_statistics(white_wine_stats,white_target)
stats_ccf = runs_statistics(ccf_stats, true_outliers=true_outliers_ccf)
stats_fire = runs_statistics(fires_stats)

stats_fire

In [ ]:
evaluate_COR(cor_labels_shuttle,shuttle_target.to_numpy(), true_outliers_shuttle)

### Sensitivity to K and o: Test on Forest Fires

In [ ]:
fires = pd.read_csv("datasets/forestfires.csv")
fires, add_parts_fires = make_categoric_partitions(fires)
fires = standardise(fires)

dots = 15
K_values = np.arange(2,dots + 2)
o_percentages = np.arange(0,0.01*dots,0.01)      

performance = np.zeros(shape = (dots,dots))
for k_index, K, in enumerate(K_values):
    for o_index, o in enumerate(o_percentages):
        performance[k_index][o_index] = COR(X = fires, K = K, o = int(len(fires)*o),r = r, n_init = n_init, 
                                            k_max=max(K_values)*2, additional_bp=add_parts_fires, seed = seed)[1]

In [ ]:
obj_min = performance.min()
rows, cols = np.where(performance == obj_min)
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(projection="3d")

o_grid, K_grid = np.meshgrid([100 * p for p in o_percentages], K_values)
ax.plot_surface(o_grid, K_grid, performance, cmap="viridis", edgecolor="k", linewidth=0.3, alpha=0.85)


for k in range(len(rows)):
    K_min = K_values[rows[k]]
    o_min = int(o_percentages[cols[k]] * 100)
    ax.scatter(o_min, K_min, obj_min, color="red", s=60, depthshade=False)
    print(f"Minimum: {obj_min:.2f} at o = {o_min}%, K = {K_min}")
ax.set_xlabel("o (%)")
ax.set_ylabel("K")
ax.set_zlabel("\nbest objective value")
ax.set_yticks(K_values)
ax.view_init(elev=25, azim=30)
plt.tight_layout()
plt.show()